# Foot Trajectory Analysis and Visualization

Tools for analyzing and visualizing foot trajectory data aligned to musical structure (downbeats, beats, subdivisions).

**Features**: Downbeat/beat/subdivision-aligned trajectories, onset detection integration, time-windowed analysis, batch processing

**Data**: Foot position CSVs (`data/logs_v1_may/` or `data/logs_v2_may/`), virtual cycle CSVs (`data/virtual_cycles/`), onset CSVs (`data/logs_v1_may/.../onset_info/`).

**Output**: Static plots (PNG), Saved to `output_static_plot/foot_trajectories/` or custom directories.

In [1]:
import os
import pickle
# import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from bvh_converter import bvh_mod
# from scipy.signal import savgol_filter

# from matplotlib.lines import Line2D
# from scipy.interpolate import interp1d

# Import trajectory plotting functions from utils
from utils_trajectory.trajectory_by_beat import plot_trajectories_by_beat
from utils_trajectory.trajectory_by_subdiv import plot_trajectories_by_subdiv
from utils_trajectory.trajectory_cycle import plot_trajectories_downbeat_window

# Import video functions from utils
# from utils_composite_video.by_time_segment import create_concat_file, concatenate_videos

In [ ]:
# file_name = "BKO_E1_D1_07_Suku"
# pickle_path = f'data/motion_data_pkl/{file_name}_T.pkl'
    
# # if os.path.isfile(pickle_path):
# with open(pickle_path, 'rb') as file:
#     motiondata_section = pickle.load(file)
# print(f"Loaded {file_name} pickle")


# Lfoot_xyzpos_data = motiondata_section["segments"]['LeftFoot']['position']
# Lfoot_zpos_data = savgol_filter(Lfoot_xyzpos_data[:,2], 60, 0)

# Rfoot_xyzpos_data = motiondata_section["segments"]['RightFoot']['position']
# Rfoot_zpos_data = savgol_filter(Rfoot_xyzpos_data[:,2], 60, 0)

## A.1 Plot Downbeat window

In [2]:
fig, ax = plot_trajectories_downbeat_window(
    file_name="BKO_E1_D2_04_Maraka",
    mode="group",
    base_path_cycles="data/virtual_cycles",
    base_path_logs="data/dance_onsets_v4_0.007_foot_jun3",
    W_start= 120, W_end= 180,
    n_beats_per_cycle=4, n_subdiv_per_beat=3, nn=8,
    use_cycles=True,
    show_gray_plots=True
)
plt.show()

## A.2 Plot trajectories by beat

In [3]:
fig, ax = plot_trajectories_by_beat(
    file_name="BKO_E1_D2_04_Maraka",
    mode="group",
    base_path_cycles="data/virtual_cycles",
    base_path_logs="data/dance_onsets_v4_0.007_foot_jun3", 
    time_segments=[(120, 180)],
    n_beats_per_cycle=4, 
    n_subdiv_per_beat=3, 
    nn=2,
    use_cycles=True,
    show_gray_plots=True
)
plt.show()

### A2.1 Batch Export

In [ ]:
mode_csv_list = os.listdir("data/subset_dance_annotation")

for mode_csv in mode_csv_list:
    file_name = mode_csv.split("_Dancers")[0]
    mode_df = pd.read_csv("data/subset_dance_annotation/" + mode_csv)

    mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
    mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
    mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

    # helper to extract a (start, end) tuple or None
    def get_segment(df, name):
        if df.empty:
            print(f"⚠️  No rows for mode '{name}', skipping.")
            return None
        return (df["Start (in sec)"].iat[0],
                df["End (in sec)"].iat[0])

    # build a dict of segments
    segments = {
        "group":      get_segment(mode_group,      "gr"),
        "individual": get_segment(mode_individual, "in"),
        "audience":   get_segment(mode_audience,   "au")
    }

    # filter out the empty ones
    tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}

    save_dir = f"output_static_plot/foot_trajectories/{file_name}"
    os.makedirs(save_dir, exist_ok=True)
    for mode, tup in tsegment.items():
        fig, ax = plot_trajectories_by_beat(
            file_name= file_name,
            mode = mode,
            base_path_cycles="data/virtual_cycles",
            base_path_logs="data/logs_v1_may",
            time_segments=[tup],
            n_beats_per_cycle=4, n_subdiv_per_beat=12, nn=2,
            use_cycles=True,
            show_gray_plots=True
        )
        fig.savefig(os.path.join(save_dir, f"{file_name}_{tup[0]}_{tup[1]}.png"))
        plt.close(fig)

## A.3 Plot trajectories by Sub division

In [4]:
fig, ax = plot_trajectories_by_subdiv(
    file_name= "BKO_E2_D4_06_Manjanin",                            # "BKO_E1_D2_04_Maraka",
    mode="group",
    base_path_cycles="data/virtual_cycles",
    base_path_logs="data/dance_onsets_v4_0.007_foot_jun3", 
    time_segments=[(231, 244)],
    n_beats_per_cycle=4, n_subdiv_per_beat=3, nn=3,
    use_cycles=True,
    show_gray_plots=False,
    subdiv_set=[2,5,8,11]    # [2,5,8,11]        [1,4,7,10]
)
plt.show()

### A3.1 Batch Export

In [ ]:
mode_csv_list = os.listdir("data/subset_dance_annotation")

for mode_csv in mode_csv_list:
    file_name = mode_csv.split("_Dancers")[0]
    mode_df = pd.read_csv("data/subset_dance_annotation/" + mode_csv)

    mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
    mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
    mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

    # helper to extract a (start, end) tuple or None
    def get_segment(df, name):
        if df.empty:
            print(f"⚠️  No rows for mode '{name}', skipping.")
            return None
        return (df["Start (in sec)"].iat[0],
                df["End (in sec)"].iat[0])

    # build a dict of segments
    segments = {
        "group":      get_segment(mode_group,      "gr"),
        "individual": get_segment(mode_individual, "in"),
        "audience":   get_segment(mode_audience,   "au")
    }

    # filter out the empty ones
    tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}

    save_dir = f"output_static_plot/foot_trajectories/{file_name}"
    os.makedirs(save_dir, exist_ok=True)
    for mode, tup in tsegment.items():
        fig, ax = plot_trajectories_by_subdiv(
            file_name= file_name,
            mode = mode,
            base_path_cycles="data/virtual_cycles",
            base_path_logs="data/logs_v1_may",
            time_segments=[tup],
            n_beats_per_cycle=4, n_subdiv_per_beat=12, nn=8,
            use_cycles=True,
            show_gray_plots=True,
            subdiv_set=[2,5,8,11]
        )
        fig.savefig(os.path.join(save_dir, f"{file_name}_{tup[0]}_{tup[1]}.png"))
        plt.close(fig)